In [12]:
import torch
import torch.nn as nn # 补全导入
import os
import torchvision
from torch.utils.data import Dataset, DataLoader
from PIL import Image

# 超参数
data_root = "./data"
img_path = "./data/img"
mask_path = "./data/mask"
batch = 20

# 1. 自定义数据类修正
class MyDataset(Dataset):
    def __init__(self, data_root):
        super().__init__()
        self.img_dir = img_path
        self.mask_dir = mask_path

        # 核心修正：排序逻辑，防止 10.png 排在 2.png 前面
        self.img_files = sorted(os.listdir(self.img_dir), key=lambda x: int(x.split('.')[0]))
        self.mask_files = sorted(os.listdir(self.mask_dir), key=lambda x: int(x.split('.')[0]))
        self.transform = torchvision.transforms.ToTensor()

    def __len__(self):
        return len(self.img_files)

    def __getitem__(self, idx):
        # 修正：直接按文件名列表读取，而不是用 idx 强行拼 .png
        img = Image.open(os.path.join(self.img_dir, self.img_files[idx])).convert("L")
        mask = Image.open(os.path.join(self.mask_dir, self.mask_files[idx])).convert("L")
        return self.transform(img), self.transform(mask)

# 2. 残差卷积块修正
class ResBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1): # 增加 stride 参数
        super().__init__()
        self.mp = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(out_channels)
        )
        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels)
            )
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        identity = x
        out = self.mp(x) # 得到卷积后的结果
        identity = self.shortcut(identity)
        out += identity # 修正变量名
        return self.relu(out)

# 3. Attention Gate (保持原样即可)
class AttentionGate(nn.Module):
    def __init__(self, F_g, F_l, F_int):
        super().__init__()
        self.W_g = nn.Sequential(
            nn.Conv2d(F_g, F_int, kernel_size=1, stride=1, padding=0, bias=True),
            nn.BatchNorm2d(F_int)
        )
        self.W_x = nn.Sequential(
            nn.Conv2d(F_l, F_int, kernel_size=1, stride=1, padding=0, bias=True),
            nn.BatchNorm2d(F_int)
        )
        self.psi = nn.Sequential(
            nn.Conv2d(F_int, 1, kernel_size=1, stride=1, padding=0, bias=True),
            nn.BatchNorm2d(1),
            nn.Sigmoid()
        )
        self.relu = nn.ReLU(inplace=True)

    def forward(self, g, x):
        g1 = self.W_g(g)
        x1 = self.W_x(x)
        psi = self.relu(g1 + x1)
        psi = self.psi(psi)
        return x * psi

# 4. Encoder / Decoder Block (修正类名引用)
class EncoderBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.res_block = ResBlock(in_channels, out_channels)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

    def forward(self, x):
        f = self.res_block(x)
        p = self.pool(f)
        return f, p

class DecoderBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.upsample = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
        self.att_gate = AttentionGate(F_g=in_ch, F_l=out_ch, F_int=out_ch // 2)
        # 统一使用 ResBlock
        self.res_block = ResBlock(in_ch + out_ch, out_ch)

    def forward(self, x, skip):
        g = self.upsample(x)
        s = self.att_gate(g=g, x=skip)
        d = torch.cat([s, g], dim=1)
        return self.res_block(d)

# 5. AttResUNet (修正初始化参数名)
class AttResUNet(nn.Module):
    def __init__(self, in_channels=1, out_channels=1):
        super().__init__()
        self.enc1 = EncoderBlock(in_channels, 64)
        self.enc2 = EncoderBlock(64, 128)
        self.enc3 = EncoderBlock(128, 256)
        self.bottleneck = ResBlock(256, 512)
        self.dec3 = DecoderBlock(512, 256)
        self.dec2 = DecoderBlock(256, 128)
        self.dec1 = DecoderBlock(128, 64)
        self.final = nn.Conv2d(64, out_channels, kernel_size=1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        f1, p1 = self.enc1(x)
        f2, p2 = self.enc2(p1)
        f3, p3 = self.enc3(p2)
        b = self.bottleneck(p3)
        d3 = self.dec3(b, f3)
        d2 = self.dec2(d3, f2)
        d1 = self.dec1(d2, f1)
        return self.sigmoid(self.final(d1))

# ==========================================
# 6. 实例化模型、数据、损失函数与优化器
# ==========================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"当前使用的设备: {device}")

# 实例化模型 (注意参数名统一)
model = AttResUNet(in_channels=1, out_channels=1).to(device)

# 实例化 DataLoader
dataset = MyDataset(data_root=data_root)
dataloader = DataLoader(dataset, batch_size=batch, shuffle=True, drop_last=False)

# 定义损失函数：网络最后一层是 Sigmoid，所以使用二值交叉熵损失 (BCELoss)
criterion = nn.BCELoss()

# 定义优化器：Adam 是目前最稳妥的选择，学习率先设为 1e-4
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

# ==========================================
# 7. 开始训练 (Training Loop)
# ==========================================
num_epochs = 50 # 你想训练的总轮数

print("开始训练...")
model.train() # 切换到训练模式（启用 BatchNorm 和 Dropout 等）

for epoch in range(num_epochs):
    epoch_loss = 0.0
    
    for step, (images, masks) in enumerate(dataloader):
        # 1. 将数据搬运到 GPU
        images = images.to(device) # [batch, 1, H, W]
        masks = masks.to(device)   # [batch, 1, H, W]
        
        # 2. 前向传播 (Forward)
        outputs = model(images)
        loss = criterion(outputs, masks)
        
        # 3. 反向传播 (Backward)
        optimizer.zero_grad() # 每次反向传播前，必须清空上一步的残余梯度
        loss.backward()       # 计算每个参数的梯度
        
        # 4. 优化更新 (Optimize)
        optimizer.step()      # 根据梯度更新网络权重
        
        epoch_loss += loss.item()
        
        # 每隔几个 Batch 打印一次进度
        if step % 5 == 0:
            print(f"Epoch [{epoch+1}/{num_epochs}], Step [{step}/{len(dataloader)}], Now Loss: {loss.item():.4f}")
            
    # 计算并打印这一轮的平均 Loss
    avg_loss = epoch_loss / len(dataloader)
    print(f"===> Epoch [{epoch+1}/{num_epochs}] Aver Loss: {avg_loss:.4f}\n")

# 训练结束后，保存模型权重
torch.save(model.state_dict(), "att_res_unet_best.pth")
print("训练完成！模型已保存为 att_res_unet_best.pth")

当前使用的设备: cuda
开始训练...


RuntimeError: The size of tensor a (96) must match the size of tensor b (97) at non-singleton dimension 2